# The Euler–Poisson–Darboux (EPD) and (2+1)-D Nonlinear Darboux Equations

This tutorial demonstrates using **`symlie`** to analyze singular hyperbolic wave equations and the Euler–Poisson–Darboux system, following **F. Güngör** (*Lie symmetry group methods for differential equations*, arXiv:1901.01543):

1. **The (1+1)-D Euler–Poisson–Darboux (EPD) Equation**: $u_{tt} + \frac{b}{t} u_t - u_{xx} = 0$
   - A four-generator finite symmetry subalgebra for general parameter $b$, in addition to linear solution superposition
   - Equivalence to the standard wave equation for $b = 0, 2$
2. **The (2+1)-D Nonlinear Darboux Wave Equation**: $\square u + \frac{b}{t} u_t + a u^k = 0$
   - Pseudo-conformal symmetry algebra in $\mathbb{R}^{1,2}$
   - Conformal Killing generators and power-law scaling

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

t, x = sp.symbols("t x", positive=True)
b = sp.symbols("b")
u = sp.Function("u")(x, t)

# 1+1 Euler-Poisson-Darboux (EPD) Equation
epd_eq = u.diff(t, 2) + (b / t) * u.diff(t) - u.diff(x, 2)
print("EPD PDE Order:", max_derivative_order(epd_eq, u, (x, t)))
sp.Eq(epd_eq, 0)

## 1. Symmetry Generators of the Linear EPD Equation

For arbitrary $b$, the EPD equation admits the displayed finite point-symmetry generators:
- Space translation: $\mathbf{v}_1 = \partial_x$
- Spacetime dilation: $\mathbf{v}_2 = x \partial_x + t \partial_t$
- Dependent-variable scaling: $\mathbf{v}_3 = u \partial_u$
- Conformal transformation: $\mathbf{v}_4 = (t^2 + x^2)\partial_x + 2tx\partial_t - b x u\partial_u$

Because the equation is linear, it additionally admits $h(x,t)\partial_u$ for every solution $h$; $u\partial_u$ itself is scaling, not solution superposition.

In [ ]:
v1 = InfinitesimalGenerator(xi=(1, 0), phi=(0,))
v2 = InfinitesimalGenerator(xi=(x, t), phi=(0,))
v3 = InfinitesimalGenerator(xi=(0, 0), phi=(u,))
v4 = InfinitesimalGenerator(xi=(t**2 + x**2, 2 * t * x), phi=(-b * x * u,))

print("v_1 (Space Translation) Invariant:", verify_generator(epd_eq, u, (x, t), v1))
print("v_2 (Spacetime Dilation) Invariant:", verify_generator(epd_eq, u, (x, t), v2))
print(
    "v_3 (Dependent-Variable Scaling) Invariant:",
    verify_generator(epd_eq, u, (x, t), v3),
)
print(
    "v_4 (Conformal Transformation) Invariant:", verify_generator(epd_eq, u, (x, t), v4)
)

## 2. Equivalence to the Standard Wave Equation ($b = 2$)

When $b = 2$, substituting $u(x, t) = \frac{1}{t} v(x, t)$ transforms the singular EPD equation $u_{tt} + \frac{2}{t} u_t - u_{xx} = 0$ into the standard wave equation $v_{tt} - v_{xx} = 0$.

In [ ]:
v_wave = sp.Function("v")(x, t)
epd_b2 = epd_eq.subs(b, 2)

# Substitute u(x, t) = v(x, t) / t
transformed = sp.simplify(epd_b2.subs(u, v_wave / t).doit())
print("Transformed Equation for v(x, t):")
display(transformed)

# Multiply by t
standard_wave = sp.simplify(transformed * t)
print("t * Transformed PDE:")
display(standard_wave)
assert sp.simplify(standard_wave - (v_wave.diff(t, 2) - v_wave.diff(x, 2))) == 0
print(
    "Confirmed: For b = 2, EPD transforms precisely to the standard 1D Wave equation!"
)

## 3. (2+1)-D Nonlinear Darboux Equation

The nonlinear wave model in (2+1) dimensions is:
$$\square u + \frac{b}{t} u_t + a u^k = 0, \qquad \square = \partial_t^2 - \partial_x^2 - \partial_y^2$$

Under scale transformations $\mathbf{d} = x\partial_x + y\partial_y + t\partial_t + \frac{2}{1-k} u\partial_u$, the equation is invariant for any power $k \neq 1$.

In [ ]:
y = sp.symbols("y")
u_3d = sp.Function("u")(x, y, t)
a, k = sp.symbols("a k")

# 2+1 Darboux wave equation
box_u = u_3d.diff(t, 2) - u_3d.diff(x, 2) - u_3d.diff(y, 2)
darboux_eq = box_u + (b / t) * u_3d.diff(t) + a * u_3d**k
print("Darboux PDE Order:", max_derivative_order(darboux_eq, u_3d, (x, y, t)))

# Dilation generator: d = x d/dx + y d/dy + t d/dt + 2/(1-k) u d/du
q_scale = 2 / (1 - k)
d_darboux = InfinitesimalGenerator(xi=(x, y, t), phi=(q_scale * u_3d,))

is_valid = verify_generator(darboux_eq, u_3d, (x, y, t), d_darboux)
print("Darboux Scale Invariance Verified:", is_valid)
assert is_valid
print("Verification: (2+1)-D Darboux equation is scale invariant!")